# Thành viên 1 – Data Cleaning
Xử lý giá trị null, phát hiện và loại bỏ anomaly trong bộ dữ liệu Home Credit.

In [1]:
import os
base_path = r"C:\Users\PC\Desktop\big data\home_credit_bigdata\data"
file_name = "application_train.csv"

if os.path.exists(os.path.join(base_path, file_name)):
    print("✅ Tìm thấy file! Bạn có thể dùng đường dẫn này.")
else:
    print("❌ Vẫn không tìm thấy. Hãy kiểm tra lại thư mục.")
    print(f"Thư mục hiện tại của bạn là: {os.getcwd()}")

✅ Tìm thấy file! Bạn có thể dùng đường dẫn này.


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import os
print("=== LOAD DATA CELL ===")
# ── Khởi tạo Spark ─────────────────────────────────────────────
spark = SparkSession.builder \
    .appName("HomeCreditDefaultRisk") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# ── Đường dẫn dữ liệu ──────────────────────────────────────────
DATA_DIR = r"C:\Users\PC\Desktop\big data\home_credit_bigdata\data"

# ── Danh sách file ─────────────────────────────────────────────
files = {
    "app_train": "application_train.csv",
    "app_test": "application_test.csv",
    "bureau": "bureau.csv",
    "bureau_bal": "bureau_balance.csv",
    "prev_app": "previous_application.csv",
    "pos_cash": "POS_CASH_balance.csv",
    "credit_card": "credit_card_balance.csv",
    "installments": "installments_payments.csv",
}

# ── Đọc dữ liệu ────────────────────────────────────────────────
dfs = {}

for name, fname in files.items():
    path = os.path.join(DATA_DIR, fname)

    print(f"\n🔍 Checking file: {path}")

    # kiểm tra file tồn tại
    if not os.path.exists(path):
        print(f"❌ File NOT FOUND: {path}")
        continue
    else:
        print("✅ File exists")

    # đọc bằng Spark
    df = spark.read.csv(
        path,
        header=True,
        inferSchema=True,
        nullValue="XNA"
    )

    dfs[name] = df

    print(f"✓ {name:15s} → {df.count():>10,} rows | {len(df.columns):>3} cols")

# ── Test truy cập dataframe ────────────────────────────────────
if "app_train" in dfs:
    print("\n📊 Sample data:")
    dfs["app_train"].show(5)

=== LOAD DATA CELL ===

🔍 Checking file: C:\Users\PC\Desktop\big data\home_credit_bigdata\data\application_train.csv
✅ File exists
✓ app_train       →    307,511 rows | 122 cols

🔍 Checking file: C:\Users\PC\Desktop\big data\home_credit_bigdata\data\application_test.csv
✅ File exists
✓ app_test        →     48,744 rows | 121 cols

🔍 Checking file: C:\Users\PC\Desktop\big data\home_credit_bigdata\data\bureau.csv
✅ File exists
✓ bureau          →  1,716,428 rows |  17 cols

🔍 Checking file: C:\Users\PC\Desktop\big data\home_credit_bigdata\data\bureau_balance.csv
✅ File exists
✓ bureau_bal      → 27,299,925 rows |   3 cols

🔍 Checking file: C:\Users\PC\Desktop\big data\home_credit_bigdata\data\previous_application.csv
✅ File exists
✓ prev_app        →  1,670,214 rows |  37 cols

🔍 Checking file: C:\Users\PC\Desktop\big data\home_credit_bigdata\data\POS_CASH_balance.csv
✅ File exists
✓ pos_cash        → 10,001,358 rows |   8 cols

🔍 Checking file: C:\Users\PC\Desktop\big data\home_credit_b

# 1. Nhóm định danh & Mục tiêu (Identifiers & Target)
#    - SK_ID_CURR:        ID duy nhất của hồ sơ vay
#    - TARGET:            Biến mục tiêu (1: khách hàng nợ xấu/khó khăn tài chính, 0: thanh toán tốt)
#
# 2. Nhóm Thông tin Khoản vay (Loan Features)
#    - NAME_CONTRACT_TYPE:    Loại hình vay (vay tiền mặt/cash loans, thấu chi/revolving loans)
#    - AMT_CREDIT:            Tổng số tiền tín dụng
#    - AMT_ANNUITY:           Số tiền phải trả định kỳ (thường theo tháng)
#    - AMT_GOODS_PRICE:       Giá trị hàng hóa dự định mua (vay tiêu dùng)
#    - NAME_TYPE_SUITE:       Người đi cùng khi nộp hồ sơ (bạn bè, gia đình, đi một mình...)
#
# 3. Nhóm Nhân khẩu học & Tài sản (Demographic & Assets)
#    - CODE_GENDER:           Giới tính
#    - FLAG_OWN_CAR / FLAG_OWN_REALTY:  Sở hữu ô tô / bất động sản (Y/N)
#    - CNT_CHILDREN:          Số con cái
#    - CNT_FAM_MEMBERS:       Số thành viên gia đình
#    - NAME_INCOME_TYPE:      Nguồn thu nhập (đi làm, kinh doanh, hưu trí, thất nghiệp...)
#    - NAME_EDUCATION_TYPE:   Trình độ học vấn cao nhất
#    - NAME_FAMILY_STATUS:    Tình trạng hôn nhân
#    - NAME_HOUSING_TYPE:     Loại hình nơi ở (ở với bố mẹ, thuê, nhà riêng...)
#
# 4. Nhóm Thời gian (Temporal Features - tính bằng ngày, số âm là tính ngược từ ngày nộp hồ sơ)
#    - DAYS_BIRTH:            Tuổi (số ngày từ sinh đến hiện tại)
#    - DAYS_EMPLOYED:         Thời gian làm việc ở vị trí hiện tại
#    - DAYS_REGISTRATION:     Thời gian đăng ký cư trú gần nhất
#    - DAYS_ID_PUBLISH:       Thời gian cấp/đổi CMND/CCCD gần nhất
#    - OWN_CAR_AGE:           Tuổi đời xe ô tô cá nhân
#
# 5. Nhóm Cờ liên lạc & Hồ sơ (Flags & Documents)
#    - FLAG_MOBIL, FLAG_EMP_PHONE, FLAG_WORK_PHONE, FLAG_PHONE, FLAG_EMAIL:
#        Các cột (0/1): khách hàng có cung cấp SĐT hoặc email không
#    - FLAG_DOCUMENT_2 đến 21:
#        Các cột (0/1): khách hàng có nộp loại hồ sơ tương ứng cho Home Credit không
#
# 6. Nhóm Điểm đánh giá bên ngoài (External Sources)
#    - EXT_SOURCE_1, EXT_SOURCE_2, EXT_SOURCE_3:
#        Điểm tín dụng tổng hợp từ nhiều nguồn ngoài (rất quan trọng dự báo nợ xấu)
#
# 7. Nhóm Thông tin nơi ở chi tiết (Building Information)
#    - Khoảng 50 cột với đuôi: _AVG (trung bình), _MODE (yếu vị), _MEDI (trung vị)
#    - Ví dụ:
#        + APARTMENTS...:          Diện tích căn hộ
#        + BASEMENTAREA...:        Diện tích tầng hầm
#        + YEARS_BEGINEXPLUATATION...: Bắt đầu sử dụng tòa nhà
#        + FLOORSMAX... / FLOORSMIN...: Số tầng cao nhất/thấp nhất
#        + WALLSMATERIAL_MODE:     Vật liệu xây tường (gạch, đá, panel...)
#        + EMERGENCYSTATE_MODE:    Nhà có trong tình trạng khẩn cấp/nguy hiểm không
#
# 8. Nhóm Quan hệ xã hội (Social Circle)
#    - OBS_30_CNT_SOCIAL_CIRCLE / OBS_60_CNT_SOCIAL_CIRCLE:
#        Số lượng bạn bè/người thân trong vòng xã hội có lịch sử nợ xấu 30/60 ngày
#    - DEF_30_CNT_SOCIAL_CIRCLE / DEF_60_CNT_SOCIAL_CIRCLE:
#        Số người trong vòng xã hội thực sự đã vỡ nợ
#
# 9. Nhóm Truy vấn thông tin tín dụng (Credit Bureau Queries)
#    - AMT_REQ_CREDIT_BUREAU_HOUR, DAY, WEEK, MON, QRT, YEAR:
#        Số lần yêu cầu kiểm tra tín dụng trong 1 giờ, 1 ngày, 1 tuần, 1 tháng, 1 quý, 1 năm gần nhất.
#        Nếu chỉ số này cao → Khách có thể đang vay nhiều nơi cùng lúc.

In [3]:
from pyspark.sql import functions as F
from collections import Counter

def eda_report(df, name, target_col=None):

    # ── [1] Shape & Schema ────────────────────────────────────────
    n_rows = df.count()
    n_cols = len(df.columns)
    print(f"\n{'='*60}")
    print(f"  EDA: {name}")
    print(f"  Shape: {n_rows:,} rows  x  {n_cols} cols")
    print(f"{'='*60}")
    df.printSchema()

    # ── [2] Sample ────────────────────────────────────────────────
    print("\n[2] SAMPLE (5 rows):")
    df.show(5, truncate=True)

    # ── [3] Data types tổng hợp ───────────────────────────────────
    print("\n[3] DATA TYPES SUMMARY:")
    type_counts = Counter(t for _, t in df.dtypes)
    for dtype, cnt in sorted(type_counts.items()):
        print(f"  {dtype:15s}: {cnt} cols")

    # ── [4] Missing values — KHÔNG dùng loop, KHÔNG dùng closure ──
    # Xây list expression trước, select 1 lần duy nhất
    print("\n[4] MISSING VALUE RATIO:")

    null_exprs = []
    for c in df.columns:
        expr = F.sum(
            F.when(F.col(c).isNull(), F.lit(1)).otherwise(F.lit(0))
        ).alias(c)
        null_exprs.append(expr)

    null_counts_row = df.agg(*null_exprs).collect()[0].asDict()

    null_data = [
        (col, cnt, round(cnt / n_rows * 100, 2))
        for col, cnt in null_counts_row.items()
        if cnt > 0
    ]
    null_data.sort(key=lambda x: x[1], reverse=True)

    if null_data:
        print(f"  Số cột có null: {len(null_data)} / {n_cols}\n")
        print(f"  {'Column':<45} {'Null Count':>12} {'Null %':>10}")
        print(f"  {'-'*45} {'-'*12} {'-'*10}")
        for col, cnt, pct in null_data[:30]:
            print(f"  {col:<45} {cnt:>12,} {pct:>9.2f}%")
    else:
        print("  >>> Không có null value nào!")

    # ── [5] Describe numeric ──────────────────────────────────────
    print("\n[5] NUMERIC STATS:")
    numeric_cols = [c for c, t in df.dtypes
                    if t in ("int", "double", "float", "bigint", "long")]

    if numeric_cols:
        # Tính min/max/mean/stddev trong 1 job, không dùng closure
        stats_exprs = []
        for c in numeric_cols[:12]:
            stats_exprs.extend([
                F.min(F.col(c)).alias(f"min__{c}"),
                F.max(F.col(c)).alias(f"max__{c}"),
                F.round(F.avg(F.col(c)), 2).alias(f"avg__{c}"),
                F.round(F.stddev(F.col(c)), 2).alias(f"std__{c}"),
            ])

        stats = df.agg(*stats_exprs).collect()[0].asDict()

        print(f"\n  {'Column':<35} {'Min':>12} {'Max':>12} "
              f"{'Mean':>12} {'Std':>12}")
        print(f"  {'-'*35} {'-'*12} {'-'*12} {'-'*12} {'-'*12}")
        for c in numeric_cols[:12]:
            mn  = stats.get(f"min__{c}", "N/A")
            mx  = stats.get(f"max__{c}", "N/A")
            avg = stats.get(f"avg__{c}", "N/A")
            std = stats.get(f"std__{c}", "N/A")
            print(f"  {c:<35} {str(mn):>12} {str(mx):>12} "
                  f"{str(avg):>12} {str(std):>12}")
    else:
        print("  Không có numeric col nào.")

    # ── [6] Categorical cols — top values ─────────────────────────
    print("\n[6] CATEGORICAL COLS (top 3 values):")
    string_cols = [c for c, t in df.dtypes if t == "string"]
    print(f"  Tổng string cols: {len(string_cols)}")

    for c in string_cols[:5]:          # chỉ xem 5 col đầu
        print(f"\n  >> {c}:")
        df.groupBy(F.col(c)) \
          .agg(F.count(F.lit(1)).alias("cnt")) \
          .orderBy(F.col("cnt").desc()) \
          .show(3, truncate=False)

    # ── [7] Target distribution ───────────────────────────────────
    if target_col and target_col in df.columns:
        print(f"\n[7] TARGET DISTRIBUTION ({target_col}):")
        df.groupBy(F.col(target_col)) \
          .agg(
              F.count(F.lit(1)).alias("count"),
              F.round(F.count(F.lit(1)) / n_rows * 100, 2).alias("pct_%")
          ) \
          .orderBy(F.col(target_col)) \
          .show()

    print(f"\n{'='*60}")
    print(f"  EDA xong: {name}")
    print(f"{'='*60}\n")

    return null_data


# ── Chạy ─────────────────────────────────────────────────────────
print(">>> Chạy EDA app_train ...")
null_train = eda_report(dfs["app_train"], "application_train", target_col="TARGET")

print(">>> Chạy EDA bureau ...")
null_bureau = eda_report(dfs["bureau"], "bureau")

>>> Chạy EDA app_train ...

  EDA: application_train
  Shape: 307,511 rows  x  122 cols
root
 |-- SK_ID_CURR: integer (nullable = true)
 |-- TARGET: integer (nullable = true)
 |-- NAME_CONTRACT_TYPE: string (nullable = true)
 |-- CODE_GENDER: string (nullable = true)
 |-- FLAG_OWN_CAR: string (nullable = true)
 |-- FLAG_OWN_REALTY: string (nullable = true)
 |-- CNT_CHILDREN: integer (nullable = true)
 |-- AMT_INCOME_TOTAL: double (nullable = true)
 |-- AMT_CREDIT: double (nullable = true)
 |-- AMT_ANNUITY: double (nullable = true)
 |-- AMT_GOODS_PRICE: double (nullable = true)
 |-- NAME_TYPE_SUITE: string (nullable = true)
 |-- NAME_INCOME_TYPE: string (nullable = true)
 |-- NAME_EDUCATION_TYPE: string (nullable = true)
 |-- NAME_FAMILY_STATUS: string (nullable = true)
 |-- NAME_HOUSING_TYPE: string (nullable = true)
 |-- REGION_POPULATION_RELATIVE: double (nullable = true)
 |-- DAYS_BIRTH: integer (nullable = true)
 |-- DAYS_EMPLOYED: integer (nullable = true)
 |-- DAYS_REGISTRATION: 

# **Đo lường kích thước dữ liệu (Inventory)**
# - Đếm tổng số dòng và số cột.
# - Thống kê các kiểu dữ liệu: bao nhiêu cột số, bao nhiêu cột chữ.

#  **Kiểm tra chất lượng dữ liệu (Data Quality)**
# - Tự động phát hiện và liệt kê các cột bị thiếu giá trị (Null).
# - Tính tỷ lệ % thiếu hụt để dễ quyết định xử lý (xóa hoặc bổ sung dữ liệu).

#  **Phân tích thống kê nhanh (Statistical Profiling)**
# - Đối với cột số: Tính giá trị nhỏ nhất, lớn nhất, trung bình, độ lệch chuẩn.
# - Đối với cột chữ: Liệt kê Top 3 giá trị phổ biến nhất (ví dụ: 3 thành phố xuất hiện nhiều nhất trong cột "Thành phố").

#  **Phân tích biến mục tiêu (Target Check)**
# - Thống kê tỷ lệ xuất hiện của từng giá trị trong biến mục tiêu (Target).
# - Quan trọng để đánh giá dữ liệu có mất cân bằng hay không, ví dụ: 90% khách hàng trả đúng hạn, 10% nợ xấu.

In [4]:
def handle_missing(df, name, null_threshold=0.5):
    """
    - Drop cột có tỷ lệ null > null_threshold
    - Impute numeric  → median
    - Impute string   → mode
    """
    n_rows = df.count()
    
    # 3a. Tìm cột cần drop
    cols_to_drop = [
        c for c in df.columns
        if df.filter(F.col(c).isNull()).count() / n_rows > null_threshold
    ]
    print(f"[{name}] Drop {len(cols_to_drop)} cột null>{null_threshold*100:.0f}%: {cols_to_drop}")
    df = df.drop(*cols_to_drop)
    
    # 3b. Phân loại còn lại
    numeric_cols  = [c for c, t in df.dtypes if t in ("int", "double", "float", "bigint")]
    string_cols   = [c for c, t in df.dtypes if t == "string"]
    
    # 3c. Impute numeric → median (dùng approxQuantile cho hiệu năng)
    for col in numeric_cols:
        null_count = df.filter(F.col(col).isNull()).count()
        if null_count > 0:
            median_val = df.approxQuantile(col, [0.5], 0.001)[0]
            df = df.fillna({col: median_val})
    
    # 3d. Impute categorical → mode
    for col in string_cols:
        null_count = df.filter(F.col(col).isNull()).count()
        if null_count > 0:
            mode_val = (df.groupBy(col)
                          .count()
                          .orderBy("count", ascending=False)
                          .first()[col])
            if mode_val:
                df = df.fillna({col: mode_val})
    
    remaining_nulls = sum(
        df.filter(F.col(c).isNull()).count() for c in df.columns
    )
    print(f"[{name}] Remaining null cells: {remaining_nulls}")
    return df

# Áp dụng cho tất cả bảng
for key in dfs:
    dfs[key] = handle_missing(dfs[key], key)

[app_train] Drop 41 cột null>50%: ['OWN_CAR_AGE', 'EXT_SOURCE_1', 'APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'ELEVATORS_AVG', 'ENTRANCES_AVG', 'FLOORSMIN_AVG', 'LANDAREA_AVG', 'LIVINGAPARTMENTS_AVG', 'LIVINGAREA_AVG', 'NONLIVINGAPARTMENTS_AVG', 'NONLIVINGAREA_AVG', 'APARTMENTS_MODE', 'BASEMENTAREA_MODE', 'YEARS_BUILD_MODE', 'COMMONAREA_MODE', 'ELEVATORS_MODE', 'ENTRANCES_MODE', 'FLOORSMIN_MODE', 'LANDAREA_MODE', 'LIVINGAPARTMENTS_MODE', 'LIVINGAREA_MODE', 'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAREA_MODE', 'APARTMENTS_MEDI', 'BASEMENTAREA_MEDI', 'YEARS_BUILD_MEDI', 'COMMONAREA_MEDI', 'ELEVATORS_MEDI', 'ENTRANCES_MEDI', 'FLOORSMIN_MEDI', 'LANDAREA_MEDI', 'LIVINGAPARTMENTS_MEDI', 'LIVINGAREA_MEDI', 'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAREA_MEDI', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE']
[app_train] Remaining null cells: 96391
[app_test] Drop 29 cột null>50%: ['OWN_CAR_AGE', 'BASEMENTAREA_AVG', 'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'ELEVA

In [5]:
def handle_anomalies(df):
    """
    Xử lý các anomaly đặc thù của Home Credit dataset.
    """

    # ── 4a. DAYS_* columns (âm = số ngày trước ngày nộp đơn) ─────────────────
    # Giá trị dương là lỗi; DAYS_EMPLOYED = 365243 là sentinel cho "không làm việc"
    days_cols = [c for c in df.columns if c.startswith("DAYS_")]
    for col in days_cols:
        if col in df.columns:
            if col == "DAYS_EMPLOYED":
                # 365243 là giá trị sentinel → thay bằng null rồi impute
                df = df.withColumn(col,
                    F.when(F.col(col) == 365243, None).otherwise(F.col(col)))
                median_val = df.approxQuantile(col, [0.5], 0.001)[0]
                df = df.fillna({col: median_val})
            # Các DAYS_ khác: giá trị dương là sai → lấy abs
            df = df.withColumn(col, F.abs(F.col(col)))

    # ── 4b. AMT_* columns — âm là vô lý ─────────────────────────────────────
    amt_cols = [c for c in df.columns if c.startswith("AMT_")]
    for col in amt_cols:
        if col in df.columns:
            df = df.withColumn(col,
                F.when(F.col(col) < 0, None).otherwise(F.col(col)))
            median_val = df.approxQuantile(col, [0.5], 0.001)[0]
            df = df.fillna({col: median_val})

    # ── 4c. CNT_CHILDREN — con số âm hoặc bất thường (>10) ──────────────────
    if "CNT_CHILDREN" in df.columns:
        df = df.withColumn("CNT_CHILDREN",
            F.when((F.col("CNT_CHILDREN") < 0) | (F.col("CNT_CHILDREN") > 10), 0)
             .otherwise(F.col("CNT_CHILDREN")))

    # ── 4d. IQR Capping cho các cột numeric quan trọng ───────────────────────
    key_cols = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE"]
    for col in key_cols:
        if col not in df.columns:
            continue
        q1, q3 = df.approxQuantile(col, [0.25, 0.75], 0.001)
        iqr = q3 - q1
        lower = q1 - 3.0 * iqr      # dùng 3×IQR thay vì 1.5× để không quá aggressive
        upper = q3 + 3.0 * iqr
        df = df.withColumn(col,
            F.when(F.col(col) < lower, lower)
             .when(F.col(col) > upper, upper)
             .otherwise(F.col(col)))
        print(f"  Capping {col}: [{lower:,.0f}, {upper:,.0f}]")

    # ── 4e. OWN_CAR_AGE — tuổi xe âm hoặc quá cao ───────────────────────────
    if "OWN_CAR_AGE" in df.columns:
        df = df.withColumn("OWN_CAR_AGE",
            F.when((F.col("OWN_CAR_AGE") < 0) | (F.col("OWN_CAR_AGE") > 100), None)
             .otherwise(F.col("OWN_CAR_AGE")))
        median_car = df.approxQuantile("OWN_CAR_AGE", [0.5], 0.001)[0]
        df = df.fillna({"OWN_CAR_AGE": median_car})

    return df

dfs["app_train"] = handle_anomalies(dfs["app_train"])
dfs["app_test"]  = handle_anomalies(dfs["app_test"])

# Kiểm tra sau xử lý
print("Sample sau anomaly handling:")
dfs["app_train"].select(
    "DAYS_BIRTH", "DAYS_EMPLOYED", "AMT_INCOME_TOTAL",
    "AMT_CREDIT", "CNT_CHILDREN"
).describe().show()

  Capping AMT_INCOME_TOTAL: [-157,500, 472,500]
  Capping AMT_CREDIT: [-1,345,950, 2,424,600]
  Capping AMT_ANNUITY: [-37,737, 88,830]
  Capping AMT_GOODS_PRICE: [-1,084,500, 2,002,500]
  Capping AMT_INCOME_TOTAL: [-225,000, 562,500]
  Capping AMT_CREDIT: [-982,440, 1,918,080]
  Capping AMT_ANNUITY: [-40,180, 95,458]
  Capping AMT_GOODS_PRICE: [-990,000, 1,845,000]
Sample sau anomaly handling:
+-------+------------------+-----------------+------------------+-----------------+-------------------+
|summary|        DAYS_BIRTH|    DAYS_EMPLOYED|  AMT_INCOME_TOTAL|       AMT_CREDIT|       CNT_CHILDREN|
+-------+------------------+-----------------+------------------+-----------------+-------------------+
|  count|            307511|           307511|            307511|           307511|             307511|
|   mean|16036.995066843137|2251.966274377177|166025.92234077805|598740.0068826806|0.41667777738032136|
| stddev| 4363.988631785563|2136.091864685651| 83063.46925080588|400966.4974805939|

In [ ]:
Bước,Mục tiêu,Kỹ thuật sử dụng
1. Chuẩn hóa Thời gian,Xử lý các cột DAYS_*,Chuyển số âm thành trị tuyệt đối (abs). Riêng lỗi hệ thống 365243 (thất nghiệp) được thay bằng Median.
2. Sửa lỗi Logic,Xử lý các cột AMT_*,"Loại bỏ các giá trị âm vô lý (thu nhập, số tiền vay < 0) và thay bằng Median."
3. Lọc nhiễu phân loại,Cột CNT_CHILDREN,Giới hạn số con trong khoảng thực tế (0 - 10 con). Các giá trị ngoài khoảng này đưa về 0.
4. Kiểm soát biên,IQR Capping,"Xác định ""hàng rào"" dữ liệu bằng công thức 3.0×IQR. Các giá trị cực đại (Outliers) bị kéo về mức trần/sàn để tránh làm lệch mô hình."
5. Xử lý xe cộ,Cột OWN_CAR_AGE,"Loại bỏ tuổi xe âm hoặc > 100 năm, sau đó điền khuyết bằng Median."

In [6]:
# ── 5a. Bureau aggregation ────────────────────────────────────────────────────
bureau_agg = dfs["bureau"].groupBy("SK_ID_CURR").agg(
    F.count("SK_ID_BUREAU")                        .alias("BUREAU_COUNT"),
    F.sum("AMT_CREDIT_SUM")                        .alias("BUREAU_CREDIT_SUM"),
    F.mean("AMT_CREDIT_SUM_DEBT")                  .alias("BUREAU_DEBT_MEAN"),
    F.sum(F.when(F.col("CREDIT_ACTIVE")=="Active",1).otherwise(0)).alias("BUREAU_ACTIVE_COUNT"),
)

# ── 5b. Previous application aggregation ─────────────────────────────────────
prev_agg = dfs["prev_app"].groupBy("SK_ID_CURR").agg(
    F.count("SK_ID_PREV")                          .alias("PREV_APP_COUNT"),
    F.mean("AMT_APPLICATION")                      .alias("PREV_AMT_APP_MEAN"),
    F.mean("AMT_CREDIT")                           .alias("PREV_AMT_CREDIT_MEAN"),
    F.sum(F.when(F.col("NAME_CONTRACT_STATUS")=="Approved",1).otherwise(0)).alias("PREV_APPROVED_COUNT"),
)

# ── 5c. POS CASH aggregation ─────────────────────────────────────────────────
pos_agg = dfs["pos_cash"].groupBy("SK_ID_CURR").agg(
    F.count("*")                                   .alias("POS_COUNT"),
    F.mean("CNT_INSTALMENT")                       .alias("POS_INSTALMENT_MEAN"),
    F.mean("SK_DPD")                               .alias("POS_DPD_MEAN"),          # days past due
)

# ── 5d. Credit card aggregation ──────────────────────────────────────────────
cc_agg = dfs["credit_card"].groupBy("SK_ID_CURR").agg(
    F.mean("AMT_BALANCE")                          .alias("CC_BALANCE_MEAN"),
    F.mean("AMT_CREDIT_LIMIT_ACTUAL")              .alias("CC_LIMIT_MEAN"),
    F.mean("SK_DPD")                               .alias("CC_DPD_MEAN"),
)

# ── 5e. Installments aggregation ─────────────────────────────────────────────
inst_agg = dfs["installments"].groupBy("SK_ID_CURR").agg(
    F.mean("NUM_INSTALMENT_VERSION")               .alias("INST_VERSION_MEAN"),
    F.mean(F.col("AMT_PAYMENT") - F.col("AMT_INSTALMENT")).alias("INST_PAYMENT_DIFF_MEAN"),
    F.sum(F.when(F.col("DAYS_ENTRY_PAYMENT") > F.col("DAYS_INSTALMENT"), 1)
           .otherwise(0))                          .alias("INST_LATE_COUNT"),
)

# ── 5f. Join tất cả vào app_train ────────────────────────────────────────────
def join_all(main_df):
    for agg_df in [bureau_agg, prev_agg, pos_agg, cc_agg, inst_agg]:
        main_df = main_df.join(agg_df, on="SK_ID_CURR", how="left")
    return main_df

app_train_final = join_all(dfs["app_train"])
app_test_final  = join_all(dfs["app_test"])

print(f"Final train shape: {app_train_final.count():,} × {len(app_train_final.columns)}")
print(f"Final test  shape: {app_test_final.count():,} × {len(app_test_final.columns)}")

Final train shape: 307,511 × 98
Final test  shape: 48,744 × 109


In [7]:
import os

# Tạo thư mục con 'processed' bên trong DATA_DIR để tách biệt dữ liệu đã làm sạch với dữ liệu thô
output_dir = os.path.join(DATA_DIR, "processed")
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

print(f"Bắt đầu lưu dữ liệu ra định dạng Parquet tại: {output_dir}")

# Đường dẫn lưu file
train_out_path = os.path.join(output_dir, "app_train_final.parquet")
test_out_path = os.path.join(output_dir, "app_test_final.parquet")

# Lưu tập train
print("Đang ghi app_train_final...")
app_train_final.write.mode("overwrite").parquet(train_out_path)
print("✅ Đã lưu thành công: app_train_final.parquet")

# Lưu tập test
print("Đang ghi app_test_final...")
app_test_final.write.mode("overwrite").parquet(test_out_path)
print("✅ Đã lưu thành công: app_test_final.parquet")

Bắt đầu lưu dữ liệu ra định dạng Parquet tại: C:\Users\PC\Desktop\big data\home_credit_bigdata\data\processed
Đang ghi app_train_final...
✅ Đã lưu thành công: app_train_final.parquet
Đang ghi app_test_final...
✅ Đã lưu thành công: app_test_final.parquet


In [10]:
# ══════════════════════════════════════════════════════════════════
#  DATA QUALITY GATE — Đọc từ file Parquet đã lưu
# ══════════════════════════════════════════════════════════════════

from pyspark.sql import functions as F
import os

# ── Load từ Parquet ────────────────────────────────────────────────
output_dir = os.path.join(DATA_DIR, "processed")

train_path = os.path.join(output_dir, "app_train_final.parquet")
test_path  = os.path.join(output_dir, "app_test_final.parquet")

print("📂 Đang load Parquet...")
print(f"  Train: {train_path}")
print(f"  Test : {test_path}")

app_train_parquet = spark.read.parquet(train_path)
app_test_parquet  = spark.read.parquet(test_path)

print(f"  ✅ Train: {app_train_parquet.count():,} rows × {len(app_train_parquet.columns)} cols")
print(f"  ✅ Test : {app_test_parquet.count():,} rows × {len(app_test_parquet.columns)} cols")


# ── Hàm kiểm tra ──────────────────────────────────────────────────
def data_quality_gate(train_df, test_df, verbose=True):
    results  = []
    all_pass = True

    def record(name, passed, detail=""):
        nonlocal all_pass
        status = "✅ PASS" if passed else "❌ FAIL"
        if not passed:
            all_pass = False
        results.append((name, status, detail))
        if verbose:
            print(f"  {status}  |  {name}")
            if detail:
                print(f"            └─ {detail}")

    print("\n" + "═"*60)
    print("  DATA QUALITY GATE  (nguồn: Parquet)")
    print("═"*60)

    train_rows = train_df.count()
    test_rows  = test_df.count()

    # ── [1] Row count ──────────────────────────────────────────────
    print("\n[1] ROW COUNT")
    record("train rows ≥ 307,000", train_rows >= 307_000, f"Actual: {train_rows:,}")
    record("test rows ≥ 48,000",   test_rows  >= 48_000,  f"Actual: {test_rows:,}")

    # ── [2] Schema — cột bắt buộc ─────────────────────────────────
    print("\n[2] REQUIRED COLUMNS")
    must_have_train = [
        "SK_ID_CURR", "TARGET",
        "DAYS_BIRTH", "DAYS_EMPLOYED",
        "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY",
        "CNT_CHILDREN",
        "BUREAU_COUNT", "PREV_APP_COUNT",
        "POS_COUNT", "CC_BALANCE_MEAN", "INST_LATE_COUNT",
    ]
    must_have_test = [c for c in must_have_train if c != "TARGET"]

    missing_train = [c for c in must_have_train if c not in train_df.columns]
    missing_test  = [c for c in must_have_test  if c not in test_df.columns]
    record("train — tất cả cột bắt buộc tồn tại",
           len(missing_train) == 0,
           f"Thiếu: {missing_train}" if missing_train else "")
    record("test  — tất cả cột bắt buộc tồn tại",
           len(missing_test)  == 0,
           f"Thiếu: {missing_test}"  if missing_test  else "")

    # ── [3] Null trong critical cols ───────────────────────────────
    print("\n[3] NULL VALUES")
    CRITICAL_COLS = [
        "SK_ID_CURR", "TARGET", "DAYS_BIRTH", "DAYS_EMPLOYED",
        "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "CNT_CHILDREN",
    ]
    for name, df in [("train", train_df), ("test", test_df)]:
        cols_to_check = [c for c in CRITICAL_COLS if c in df.columns]
        null_exprs    = [F.sum(F.col(c).isNull().cast("int")).alias(c)
                         for c in cols_to_check]
        null_row  = df.agg(*null_exprs).collect()[0].asDict()
        null_cols = {k: v for k, v in null_row.items() if v and v > 0}
        record(f"{name} — không có null trong critical cols",
               len(null_cols) == 0,
               f"Còn null: {null_cols}" if null_cols else "")

    # ── [4] DAYS_* >= 0 ───────────────────────────────────────────
    print("\n[4] DAYS_* >= 0")
    for col in [c for c in train_df.columns if c.startswith("DAYS_")]:
        neg = train_df.filter(F.col(col) < 0).count()
        record(f"train.{col} >= 0",
               neg == 0,
               f"{neg:,} giá trị âm còn sót" if neg else "")

    # ── [5] AMT_* >= 0 ────────────────────────────────────────────
    print("\n[5] AMT_* >= 0")
    for name, df in [("train", train_df), ("test", test_df)]:
        for col in ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE"]:
            if col not in df.columns:
                continue
            neg = df.filter(F.col(col) < 0).count()
            record(f"{name}.{col} >= 0",
                   neg == 0,
                   f"{neg:,} giá trị âm còn sót" if neg else "")

    # ── [6] CNT_CHILDREN ∈ [0, 10] ────────────────────────────────
    print("\n[6] CNT_CHILDREN ∈ [0, 10]")
    for name, df in [("train", train_df), ("test", test_df)]:
        if "CNT_CHILDREN" not in df.columns:
            continue
        bad = df.filter(
            (F.col("CNT_CHILDREN") < 0) | (F.col("CNT_CHILDREN") > 10)
        ).count()
        record(f"{name}.CNT_CHILDREN ∈ [0,10]",
               bad == 0,
               f"{bad:,} giá trị ngoài khoảng" if bad else "")

    # ── [7] DAYS_EMPLOYED — không còn sentinel 365243 ─────────────
    print("\n[7] DAYS_EMPLOYED — không còn sentinel 365243")
    for name, df in [("train", train_df), ("test", test_df)]:
        if "DAYS_EMPLOYED" not in df.columns:
            continue
        sentinel = df.filter(F.col("DAYS_EMPLOYED") == 365243).count()
        record(f"{name}.DAYS_EMPLOYED ≠ 365243",
               sentinel == 0,
               f"{sentinel:,} dòng vẫn còn sentinel" if sentinel else "")

    # ── [8] TARGET hợp lệ ─────────────────────────────────────────
    print("\n[8] TARGET DISTRIBUTION")
    if "TARGET" in train_df.columns:
        invalid = train_df.filter(~F.col("TARGET").isin([0, 1])).count()
        record("train.TARGET chỉ chứa {0, 1}",
               invalid == 0,
               f"{invalid:,} giá trị không hợp lệ" if invalid else "")

        dist     = train_df.groupBy("TARGET").count().collect()
        pct_1    = {r["TARGET"]: r["count"] for r in dist}.get(1, 0) / train_rows * 100
        record("Tỷ lệ TARGET=1 ∈ [5%, 20%]",
               5.0 <= pct_1 <= 20.0,
               f"TARGET=1: {pct_1:.2f}%")

    # ── [9] Outlier AMT_INCOME_TOTAL ──────────────────────────────
    print("\n[9] OUTLIER CHECK (AMT_INCOME_TOTAL)")
    if "AMT_INCOME_TOTAL" in train_df.columns:
        q1, q3  = train_df.approxQuantile("AMT_INCOME_TOTAL", [0.25, 0.75], 0.01)
        upper   = q3 + 3.0 * (q3 - q1)
        extreme = train_df.filter(F.col("AMT_INCOME_TOTAL") > upper * 1.01).count()
        record("train.AMT_INCOME_TOTAL — không còn outlier cực đoan",
               extreme == 0,
               f"{extreme:,} dòng vượt ngưỡng {upper:,.0f}" if extreme else "")

    # ── [10] Aggregated cols có dữ liệu ───────────────────────────
    print("\n[10] AGGREGATED COLUMNS — không all-null")
    for col in ["BUREAU_COUNT", "PREV_APP_COUNT", "POS_COUNT",
                "CC_BALANCE_MEAN", "INST_LATE_COUNT"]:
        if col not in train_df.columns:
            continue
        non_null = train_df.filter(F.col(col).isNotNull()).count()
        record(f"train.{col} có dữ liệu",
               non_null > 0,
               f"non-null: {non_null:,}")

    # ── Tổng kết ───────────────────────────────────────────────────
    n_pass = sum(1 for _, s, _ in results if "PASS" in s)
    n_fail = sum(1 for _, s, _ in results if "FAIL" in s)
    print("\n" + "═"*60)
    print(f"  KẾT QUẢ: {n_pass} PASS  |  {n_fail} FAIL  |  Tổng {len(results)} checks")
    if all_pass:
        print("  🎉 FILE PARQUET ĐÃ SẠCH — SẴN SÀNG QUA FEATURE ENGINEERING!")
    else:
        print("  ⚠️  CÒN VẤN ĐỀ — Xem các check FAIL ở trên để fix.")
    print("═"*60 + "\n")

    return all_pass


# ── Chạy ──────────────────────────────────────────────────────────
is_clean = data_quality_gate(app_train_parquet, app_test_parquet)

📂 Đang load Parquet...
  Train: C:\Users\PC\Desktop\big data\home_credit_bigdata\data\processed\app_train_final.parquet
  Test : C:\Users\PC\Desktop\big data\home_credit_bigdata\data\processed\app_test_final.parquet
  ✅ Train: 307,511 rows × 98 cols
  ✅ Test : 48,744 rows × 109 cols

════════════════════════════════════════════════════════════
  DATA QUALITY GATE  (nguồn: Parquet)
════════════════════════════════════════════════════════════

[1] ROW COUNT
  ✅ PASS  |  train rows ≥ 307,000
            └─ Actual: 307,511
  ✅ PASS  |  test rows ≥ 48,000
            └─ Actual: 48,744

[2] REQUIRED COLUMNS
  ✅ PASS  |  train — tất cả cột bắt buộc tồn tại
  ✅ PASS  |  test  — tất cả cột bắt buộc tồn tại

[3] NULL VALUES
  ✅ PASS  |  train — không có null trong critical cols
  ✅ PASS  |  test — không có null trong critical cols

[4] DAYS_* >= 0
  ✅ PASS  |  train.DAYS_BIRTH >= 0
  ✅ PASS  |  train.DAYS_EMPLOYED >= 0
  ✅ PASS  |  train.DAYS_REGISTRATION >= 0
  ✅ PASS  |  train.DAYS_ID_PUBLISH 

In [9]:
# ══════════════════════════════════════════════════════════════════
#  DATA QUALITY GATE — Chạy sau khi xử lý, trước Feature Engineering
#  Kiểm tra: app_train_final và app_test_final
# ══════════════════════════════════════════════════════════════════

from pyspark.sql import functions as F

def data_quality_gate(train_df, test_df, verbose=True):
    """
    Chạy toàn bộ kiểm tra chất lượng dữ liệu.
    Trả về True nếu TẤT CẢ check đều PASS, False nếu có bất kỳ FAIL nào.
    """

    results   = []   # list of (check_name, status, detail)
    all_pass  = True

    def record(name, passed, detail=""):
        nonlocal all_pass
        status = "✅ PASS" if passed else "❌ FAIL"
        if not passed:
            all_pass = False
        results.append((name, status, detail))
        if verbose:
            print(f"  {status}  |  {name}")
            if detail:
                print(f"            └─ {detail}")

    print("\n" + "═"*60)
    print("  DATA QUALITY GATE")
    print("═"*60)

    # ── [1] Row count không bị mất ─────────────────────────────────
    print("\n[1] ROW COUNT")
    train_rows = train_df.count()
    test_rows  = test_df.count()
    record("train rows ≥ 307,000",  train_rows >= 307_000,
           f"Actual: {train_rows:,}")
    record("test rows ≥ 48,000",    test_rows  >= 48_000,
           f"Actual: {test_rows:,}")

    # ── [2] Schema — các cột bắt buộc phải có ─────────────────────
    print("\n[2] REQUIRED COLUMNS")
    must_have_train = [
        "SK_ID_CURR", "TARGET",
        "DAYS_BIRTH", "DAYS_EMPLOYED",
        "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY",
        "CNT_CHILDREN",
        # aggregated columns từ bước join
        "BUREAU_COUNT", "PREV_APP_COUNT",
        "POS_COUNT", "CC_BALANCE_MEAN", "INST_LATE_COUNT",
    ]
    must_have_test = [c for c in must_have_train if c != "TARGET"]

    missing_train = [c for c in must_have_train if c not in train_df.columns]
    missing_test  = [c for c in must_have_test  if c not in test_df.columns]
    record("train — tất cả cột bắt buộc tồn tại",
           len(missing_train) == 0,
           f"Thiếu: {missing_train}" if missing_train else "")
    record("test  — tất cả cột bắt buộc tồn tại",
           len(missing_test)  == 0,
           f"Thiếu: {missing_test}"  if missing_test  else "")

    # ── [3] Null còn sót lại ───────────────────────────────────────
    print("\n[3] NULL VALUES")
    CRITICAL_COLS = [
        "SK_ID_CURR", "TARGET", "DAYS_BIRTH", "DAYS_EMPLOYED",
        "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "CNT_CHILDREN",
    ]

    for name, df in [("train", train_df), ("test", test_df)]:
        cols_to_check = [c for c in CRITICAL_COLS if c in df.columns]
        null_exprs    = [F.sum(F.col(c).isNull().cast("int")).alias(c)
                         for c in cols_to_check]
        null_row = df.agg(*null_exprs).collect()[0].asDict()
        null_cols = {k: v for k, v in null_row.items() if v and v > 0}
        record(f"{name} — không có null trong critical cols",
               len(null_cols) == 0,
               f"Còn null: {null_cols}" if null_cols else "")

    # ── [4] DAYS_* phải không âm (đã abs) ─────────────────────────
    print("\n[4] DAYS_* >= 0 (đã chuyển abs)")
    days_cols_train = [c for c in train_df.columns if c.startswith("DAYS_")]
    for col in days_cols_train:
        neg_count = train_df.filter(F.col(col) < 0).count()
        record(f"train.{col} >= 0",
               neg_count == 0,
               f"{neg_count:,} giá trị âm còn sót" if neg_count else "")

    # ── [5] AMT_* phải không âm ────────────────────────────────────
    print("\n[5] AMT_* >= 0")
    amt_cols = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE"]
    for name, df in [("train", train_df), ("test", test_df)]:
        for col in amt_cols:
            if col not in df.columns:
                continue
            neg = df.filter(F.col(col) < 0).count()
            record(f"{name}.{col} >= 0",
                   neg == 0,
                   f"{neg:,} giá trị âm còn sót" if neg else "")

    # ── [6] CNT_CHILDREN ∈ [0, 10] ────────────────────────────────
    print("\n[6] CNT_CHILDREN ∈ [0, 10]")
    for name, df in [("train", train_df), ("test", test_df)]:
        if "CNT_CHILDREN" not in df.columns:
            continue
        bad = df.filter(
            (F.col("CNT_CHILDREN") < 0) | (F.col("CNT_CHILDREN") > 10)
        ).count()
        record(f"{name}.CNT_CHILDREN ∈ [0,10]",
               bad == 0,
               f"{bad:,} giá trị ngoài khoảng" if bad else "")

    # ── [7] DAYS_EMPLOYED không còn sentinel 365243 ────────────────
    print("\n[7] DAYS_EMPLOYED — không còn sentinel 365243")
    for name, df in [("train", train_df), ("test", test_df)]:
        if "DAYS_EMPLOYED" not in df.columns:
            continue
        sentinel = df.filter(F.col("DAYS_EMPLOYED") == 365243).count()
        record(f"{name}.DAYS_EMPLOYED ≠ 365243",
               sentinel == 0,
               f"{sentinel:,} dòng vẫn còn sentinel" if sentinel else "")

    # ── [8] TARGET distribution hợp lệ (chỉ 0 và 1) ──────────────
    print("\n[8] TARGET DISTRIBUTION (chỉ train)")
    if "TARGET" in train_df.columns:
        invalid_target = train_df.filter(
            ~F.col("TARGET").isin([0, 1])
        ).count()
        record("train.TARGET chỉ chứa {0, 1}",
               invalid_target == 0,
               f"{invalid_target:,} giá trị không hợp lệ" if invalid_target else "")

        dist = train_df.groupBy("TARGET").count().collect()
        dist_dict = {row["TARGET"]: row["count"] for row in dist}
        pct_1 = dist_dict.get(1, 0) / train_rows * 100
        record("Tỷ lệ TARGET=1 ∈ [5%, 20%] (imbalanced nhưng hợp lý)",
               5.0 <= pct_1 <= 20.0,
               f"TARGET=1: {pct_1:.2f}%")

    # ── [9] IQR — AMT_INCOME_TOTAL không còn outlier cực đoan ──────
    print("\n[9] OUTLIER CHECK (AMT_INCOME_TOTAL)")
    if "AMT_INCOME_TOTAL" in train_df.columns:
        q1, q3 = train_df.approxQuantile("AMT_INCOME_TOTAL", [0.25, 0.75], 0.01)
        iqr     = q3 - q1
        upper   = q3 + 3.0 * iqr
        extreme = train_df.filter(F.col("AMT_INCOME_TOTAL") > upper * 1.01).count()
        record("train.AMT_INCOME_TOTAL — không còn outlier cực đoan (>3×IQR)",
               extreme == 0,
               f"{extreme:,} dòng vượt ngưỡng {upper:,.0f}" if extreme else "")

    # ── [10] Feature columns từ aggregation có dữ liệu ─────────────
    print("\n[10] AGGREGATED FEATURE COLUMNS — không all-null")
    agg_cols = ["BUREAU_COUNT", "PREV_APP_COUNT", "POS_COUNT",
                "CC_BALANCE_MEAN", "INST_LATE_COUNT"]
    for col in agg_cols:
        if col not in train_df.columns:
            continue
        non_null = train_df.filter(F.col(col).isNotNull()).count()
        record(f"train.{col} có ít nhất 1 giá trị",
               non_null > 0,
               f"non-null count: {non_null:,}")

    # ══ TỔNG KẾT ═══════════════════════════════════════════════════
    print("\n" + "═"*60)
    n_pass = sum(1 for _, s, _ in results if "PASS" in s)
    n_fail = sum(1 for _, s, _ in results if "FAIL" in s)
    print(f"  KẾT QUẢ: {n_pass} PASS  |  {n_fail} FAIL  |  Tổng {len(results)} checks")
    if all_pass:
        print("  🎉 DỮ LIỆU ĐÃ SẠCH — SẴN SÀNG QUA FEATURE ENGINEERING!")
    else:
        print("  ⚠️  CÒN VẤN ĐỀ — Hãy xem lại các check FAIL ở trên trước khi tiếp tục.")
    print("═"*60 + "\n")

    return all_pass


# ── Chạy gate ─────────────────────────────────────────────────────
is_clean = data_quality_gate(app_train_final, app_test_final)


════════════════════════════════════════════════════════════
  DATA QUALITY GATE
════════════════════════════════════════════════════════════

[1] ROW COUNT
  ✅ PASS  |  train rows ≥ 307,000
            └─ Actual: 307,511
  ✅ PASS  |  test rows ≥ 48,000
            └─ Actual: 48,744

[2] REQUIRED COLUMNS
  ✅ PASS  |  train — tất cả cột bắt buộc tồn tại
  ✅ PASS  |  test  — tất cả cột bắt buộc tồn tại

[3] NULL VALUES
  ✅ PASS  |  train — không có null trong critical cols
  ✅ PASS  |  test — không có null trong critical cols

[4] DAYS_* >= 0 (đã chuyển abs)
  ✅ PASS  |  train.DAYS_BIRTH >= 0
  ✅ PASS  |  train.DAYS_EMPLOYED >= 0
  ✅ PASS  |  train.DAYS_REGISTRATION >= 0
  ✅ PASS  |  train.DAYS_ID_PUBLISH >= 0
  ✅ PASS  |  train.DAYS_LAST_PHONE_CHANGE >= 0

[5] AMT_* >= 0
  ✅ PASS  |  train.AMT_INCOME_TOTAL >= 0
  ✅ PASS  |  train.AMT_CREDIT >= 0
  ✅ PASS  |  train.AMT_ANNUITY >= 0
  ✅ PASS  |  train.AMT_GOODS_PRICE >= 0
  ✅ PASS  |  test.AMT_INCOME_TOTAL >= 0
  ✅ PASS  |  test.AMT_CRED